# Assignment 2: train on Colab

Runs the unchanged `train.py` on a Colab GPU and saves the weights, artifacts and training run
to Google Drive (and as a zip download).

Before you start:

1. **Runtime > Change runtime type > T4 GPU** (or any GPU).
2. Add your Roboflow key as a Colab secret: click the key icon in the left sidebar,
   add `ROBOFLOW_API_KEY`, paste the key and switch on *Notebook access*.
   Never paste the key into a cell.
3. Have `train.py`, `ppe_decision.py` and `api_key.py` from the repo ready to upload.

In [ ]:
!nvidia-smi -L

In [ ]:
%pip install -q "ultralytics>=8.4,<8.5" "roboflow>=1.7" "opencv-python-headless>=4.10"

Upload `train.py`, `ppe_decision.py` and `api_key.py` (select all three at once).

In [ ]:
import os
from pathlib import Path
from google.colab import files

WORK = Path("/content/daiist-assignment-2")
WORK.mkdir(exist_ok=True)
os.chdir(WORK)

NEEDED = {"train.py", "ppe_decision.py", "api_key.py"}
missing = {f for f in NEEDED if not (WORK / f).exists()}
if missing:
    for name, data in files.upload().items():
        (WORK / name).write_bytes(data)
missing = {f for f in NEEDED if not (WORK / f).exists()}
assert not missing, f"Still missing: {missing}"
print(sorted(p.name for p in WORK.iterdir()))

In [ ]:
from google.colab import userdata

# api_key.load_api_key() checks this environment variable first, so train.py needs no changes.
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

Mount Drive so the results survive the runtime shutting down.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/daiist-assignment-2-output")
OUT.mkdir(parents=True, exist_ok=True)

Train and evaluate. Set `EPOCHS` to `"1"` first if you want a quick smoke test;
the full 50-epoch run takes roughly 30 to 60 minutes on a T4.

In [ ]:
os.environ["EPOCHS"] = "50"
!python train.py

Copy the outputs to Drive and download them as a zip.

In [ ]:
import shutil

for name in ("weights", "artifacts", "runs"):
    if (WORK / name).exists():
        shutil.copytree(WORK / name, OUT / name, dirs_exist_ok=True)
print("Saved to", OUT)

# Only the parts you need back locally: the weights and the dashboard artifacts.
shutil.make_archive("/content/weights", "zip", root_dir=WORK, base_dir="weights")
shutil.make_archive("/content/artifacts", "zip", root_dir=WORK, base_dir="artifacts")
files.download("/content/weights.zip")
files.download("/content/artifacts.zip")

Back on your laptop, unzip both into the repo root so you get `weights/ppe_yolo11n.pt` and
`artifacts/...` in place, then run `uv run python main.py app`.